## Shared configuration for the Advanced ML Concepts notebooks
Run automatically by each notebook with `%run ./Includes/_config`. It creates a personal schema, a volume for files,
an MLflow experiment, and a `diabetes` table, then prints the names every notebook uses.

In [ ]:
import os, re
import pyspark.sql.functions as F
import mlflow

CATALOG = "swsacademy"
DIABETES_CSV = "/Volumes/swsacademy_cdc_diabetes/v01/cdc-diabetes/diabetes_binary_5050split_BRFSS2015.csv"

user = spark.sql("SELECT current_user() AS u").first()["u"]
slug = re.sub(r"[^a-z0-9_]", "_", user.split("@")[0].lower())
SCHEMA = f"adv_concepts_{slug}"

try:
    spark.sql(f"DESCRIBE CATALOG {CATALOG}")
    dbutils.fs.ls(DIABETES_CSV)
except Exception as e:
    raise RuntimeError(
        "The swsacademy catalog or the Diabetes dataset is missing. Run '00 - SWSAcademy Admin Setup' in "
        "'2 - swsacademy-advanced-machine-learning-operations' once, then run this notebook again."
    ) from e

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.artifacts")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/artifacts"

# Serverless needs Unity Catalog volume folders as temporary space for Spark ML
# (model saving through MLflow, and DataFrame caching during cross-validation)
os.environ["MLFLOW_DFS_TMP"] = f"{VOLUME}/mlflow_tmp"
os.environ["SPARKML_TEMP_DFS_PATH"] = f"{VOLUME}/sparkml_tmp"
mlflow.set_registry_uri("databricks-uc")
EXPERIMENT = f"/Users/{user}/SWSAcademy_Advanced_ML_Concepts"
mlflow.set_experiment(EXPERIMENT)

TABLE = f"{CATALOG}.{SCHEMA}.diabetes"
if not spark.catalog.tableExists(TABLE):
    raw = spark.read.csv(DIABETES_CSV, header=True, inferSchema=True)
    for c in raw.columns:
        raw = raw.withColumn(c, F.col(c).cast("double"))
    raw.withColumn("id", F.monotonically_increasing_id()).write.mode("overwrite").saveAsTable(TABLE)

LABEL = "Diabetes_binary"
FEATURES = [c for c in spark.table(TABLE).columns if c not in (LABEL, "id")]

print(f"User:        {user}")
print(f"Schema:      {CATALOG}.{SCHEMA}")
print(f"Table:       {TABLE}  ({spark.table(TABLE).count():,} rows, {len(FEATURES)} features)")
print(f"Volume:      {VOLUME}")
print(f"Experiment:  {EXPERIMENT}")